# Step 2: Filter notebook
Ce notebook filtre les attributs et uniforme le format afin qu'ils soient prêt à être intégré à l'indice. 
Les filtres sont évalués au cas par cas selon la pertinence.

In [ ]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
import rasterio
import numpy as np
import os
import folium
import json
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from feature_to_network import *

## Imports

#### Import des segments

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # bike ONLY in the notebook, go to 2_1_Filter_features_walk.ipynb for walk

input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-2"

save_filtered_attributes = True
attribute_source_path = f"source_path_{territory}"
file_name = f"file_name_{territory}"  # column name in attributs_info excel

# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad bike segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


In [ ]:
import geopandas as gpd
from shapely.geometry.base import BaseGeometry

# Clean geometries safely
def clean_any_geom(geom):
    if geom is None:
        return None
    if not isinstance(geom, BaseGeometry):
        return None
    if geom.is_valid:
        return geom
    try:
        fixed = geom.buffer(0)
        if fixed is None or fixed.is_empty:
            return geom
        return fixed
    except Exception:
        return geom
        




# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")


#### Import des attributs 

In [ ]:
# Classeur des attributs vélo (mettre à jour le nom si le fichier WIP est renommé)
ATTRIBUTS_INFO_FILE = "attributs_info_bike_WIP.xlsx"
attributs_info = pd.read_excel(f"{input_file_path}/attributs/{ATTRIBUTS_INFO_FILE}", sheet_name="attributs_info")

# include_in_index_<territoire> (ancien format : include_in_index). Excel peut lire
# ces colonnes en booléens ou en texte ('true'/'false') : on normalise.
include = f"include_in_index_{territory}" if f"include_in_index_{territory}" in attributs_info.columns else "include_in_index"
attributs_info = attributs_info[attributs_info[include].astype(str).str.strip().str.lower().ne("false")]

attributs_info

In [ ]:
attributs_info

**Attributs OSM**

**Attribut stationnement velo**


Amélioration: donner un meilleur score selon la capacité, présence abris ou non

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_velo'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

# --- Capacity cleaning (handles empty strings, None, non-numeric) ---
gdf["capacity_num"] = pd.to_numeric(
    gdf["capacity"].astype(str).str.strip().replace({"": None, "nan": None, "None": None}),
    errors="coerce"
)

# --- Score using capacity, fallback to 1 ---
gdf["score"] = gdf["capacity_num"].fillna(1)

# (optional) avoid zero/negative capacities becoming weird scores
gdf.loc[gdf["score"] <= 0, "score"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut location**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'location'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut zone apaisée**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[(gdf.speed_kph > 5) & (gdf.speed_kph < 30), 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Atttribut Vitesse motorisée**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse_motorisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.speed_kph>25, 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Contre-sens cyclable**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'sens_inverse'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.TYPE_AMEN=="Contresens", 'filtered'] = 1
gdf.loc[gdf.TYPE_AMEN=="Dérogation 2R", 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Giratoire**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'giratoire'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut revêtement**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'revetement'


###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
surface_score = {
    "asphalt": 1,
    "concrete": 1,
    "concrete:plates": 1,
    "concrete:lanes": 1,
    "paved": 0,
    "paving_stones": 0,
    "sett": 0,
    "metal": 0,
    "wood": 0,

    "fine_gravel": 0,
    "compacted": 0,
    "gravel": 0,
    "pebblestone": 0,

    "ground": 0,
    "dirt": 0,
    "earth": 0,
    "grass": 0,
    "grass_paver": 0,
    "sand": 0,
    "rock": 0,
    "unpaved": 0,
    "woodchips": 0,
}


gdf["surface_score"] = gdf["surface"].map(surface_score)
gdf.loc[gdf["surface_score"] == 1, "filtered"] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Aménité**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")
# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
rez_actif_set = {
"restaurant","cafe","bar","pub","fast_food","ice_cream","biergarten",
"bank","atm","bureau_de_change",
"cinema","theatre","nightclub","casino",
"marketplace","vending_machine"
}

gdf.loc[gdf["amenity"].isin(rez_actif_set), "filtered"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


**Attribut borne réparation**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'borne_reparation'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut éclairage**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eclairage'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.lit=='yes', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Confort thermique**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----

attribute = 'temperature'
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Keep native raster resolution for parquet export
    rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'temperature': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Lac et cours d'eau**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eau'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)


**Attribut Canopée**

GG : couverture du sol SITG (`SIPV_MN_CARTO_GG.gpkg`, colonne `vegetation`), forêts et végétation arborée uniquement, découpée au périmètre Grand Genève (même logique que 2_1 walk). GE : ancienne couche OSM.

In [ ]:
import difflib
import importlib.util
import pyogrio
import shapely


def load_land_cover_classes(gpkg_path, layer, class_column, classes, perimetre_path):
    """
    Lit dans un GeoPackage de couverture du sol (structure SIPV_MN_CARTO_GG) les
    polygones des classes demandées, découpés au périmètre Grand Genève.
    Retourne une GeoDataFrame (colonne `classe`) en operation_crs.
    """
    # Couche à lire
    layers = pyogrio.list_layers(gpkg_path)
    if layer is None:
        if len(layers) != 1:
            raise ValueError(f"Plusieurs couches dans {gpkg_path} : {layers[:, 0].tolist()} — précisez-en une.")
        layer = layers[0][0]

    # Vérification des classes (lecture de la seule colonne, sans géométries)
    classes_df = pyogrio.read_dataframe(
        gpkg_path,
        sql=f'SELECT "{class_column}" AS classe, COUNT(*) AS n FROM "{layer}" GROUP BY "{class_column}" ORDER BY n DESC',
        read_geometry=False,
    )
    available = classes_df["classe"].dropna().astype(str).tolist()
    unknown = [c for c in classes if c not in available]
    for c in unknown:
        print(f"⚠ classe introuvable : {c!r} — proches : {difflib.get_close_matches(c, available, n=3, cutoff=0.5)}")
    if unknown:
        print("Classes disponibles :", available)
        raise ValueError("Corrigez la liste des classes avec des valeurs exactes de la liste ci-dessus.")

    # Périmètre GG (CRS métrique) et son emprise dans le CRS du gpkg (préfiltre spatial)
    perimetre = gpd.read_file(perimetre_path).to_crs(operation_crs)
    perimetre_geom = shapely.make_valid(perimetre.union_all())
    gpkg_crs = pyogrio.read_info(gpkg_path, layer=layer)["crs"]
    bbox = tuple(gpd.GeoSeries([perimetre_geom], crs=operation_crs).to_crs(gpkg_crs).total_bounds)

    # Lecture filtrée dans GDAL : classes choisies + emprise du périmètre
    quoted = ", ".join("'" + c.replace("'", "''") + "'" for c in classes)
    gdf = pyogrio.read_dataframe(
        gpkg_path, layer=layer, columns=[class_column],
        where=f'"{class_column}" IN ({quoted})',
        bbox=bbox,
        use_arrow=importlib.util.find_spec("pyarrow") is not None,
    )
    gdf = gdf.rename(columns={class_column: "classe"})
    print(f"  {len(gdf):,} polygones lus pour {len(classes)} classes")

    # Nettoyage (valides, polygonales, non vides) puis découpe au périmètre
    def _clean(g):
        g["geometry"] = shapely.make_valid(g.geometry.values)
        g = g.explode(index_parts=False, ignore_index=True)
        return g[g.geom_type.isin(["Polygon", "MultiPolygon"]) & ~g.is_empty].copy()

    gdf = _clean(gdf).to_crs(operation_crs)
    n_avant = len(gdf)
    gdf = _clean(gpd.clip(gdf, perimetre_geom))
    print(f"  Découpe au périmètre GG : {n_avant:,} -> {len(gdf):,} polygones")
    print(gdf.groupby("classe").geometry.apply(lambda g: g.area.sum() / 1e4).rename("surface_ha").round(1))
    return gdf


# Réglages communs aux couches de couverture du sol
LAND_COVER_LAYER = None          # None = détection auto s'il n'y a qu'une couche
LAND_COVER_CLASS_COLUMN = "vegetation"
PERIMETRE_GG = f"{input_file_path}/network_agreg/AGGLO_PERIMETRE_AVEC_LAC-SHP"


In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'canopee'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

if territory == 'GG':
    ###----change here----------------------
    # Couverture du sol SITG (chemin et fichier lus dans l'Excel) : végétation arborée uniquement
    CANOPEE_CLASSES = [
        "Forêts fermées de conifères",
        "Forêts fermées de feuillus",
        "Forêts ouvertes",
        "Forêts humides",
        "végétation urbaine",
        "Végétation arborée perturbée",
    ]
    ###--------------------------------------
    gdf = load_land_cover_classes(
        f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
        LAND_COVER_LAYER, LAND_COVER_CLASS_COLUMN, CANOPEE_CLASSES, PERIMETRE_GG,
    ).to_crs(target_crs)
else:
    # GE : ancienne couche OSM (landuse=forest, natural=wood)
    gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
    gdf = gdf.to_crs(target_crs)
    gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
    gdf = gdf[gdf.geometry.notna()].copy()

# Filter
gdf['filtered'] = 0
###----change here----------------------
gdf['filtered'] = 1   # la sélection par classe est déjà faite à la lecture
###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut Environnement ouvert naturel**

GG : même couverture du sol que la canopée, classes prairies, cultures, vigne et verger. Pas de routine pour GE.

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'environnement_ouvert_naturel'
###--------------------

if territory != 'GG':
    print(f"{attribute} : pas de routine pour {territory}")
else:
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    print(f"Processing attribute: {attribute}")

    ###----change here----------------------
    # Même couverture du sol que la canopée : milieux ouverts naturels et agricoles
    ENV_OUVERT_CLASSES = [
        "Prairies humides",
        "Prairies sèches",
        "Prairies extensives",
        "Prairies d'altitudes",
        "Cultures et prairies intensives",
        "Vigne",
        "Verger",
    ]
    ###--------------------------------------
    gdf = load_land_cover_classes(
        f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
        LAND_COVER_LAYER, LAND_COVER_CLASS_COLUMN, ENV_OUVERT_CLASSES, PERIMETRE_GG,
    ).to_crs(target_crs)

    # Filter
    gdf['filtered'] = 0
    ###----change here----------------------
    gdf['filtered'] = 1   # la sélection par classe est déjà faite à la lecture
    ###--------------------------------------
    print("Filters applied")

    # Count and proportion
    print(f"Proportion of features {attribute} kept after filtering:")
    print(gdf.filtered.value_counts(normalize=True))

    # Save
    save(save_filtered_attributes, row, gdf, attribute)

**Attribut pollution air**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'air'

###--------------------

    
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Keep air sampling denser than before, but lighter than temperature
    rows, cols = rows[::1], cols[::1]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # ✅ Conversion en µmol/m² pour éviter les underflows d'affichage
    temp_values_converted = temp_values * 1e6  # mol/m² → µmol/m²
    
    gdf = gpd.GeoDataFrame({
        'no_2': temp_values_converted,   # valeurs lisibles ex: 18 à 40 µmol/m²
        'no_2_raw': temp_values,         # valeurs originales conservées si besoin
        'filtered': 1,
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['no_2'].describe())

# Save
print(f"Saving {attribute}: can take up to 3mn")
save(save_filtered_attributes, row, gdf, attribute)

**Attribut piste**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'piste'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")


gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf['infra_bike'] == 'piste_cyclable', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut bande**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bande'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf['infra_bike'] == 'bande_cyclable', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut environnement agréable (alentours)**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'alentours'


###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
landuse_agreable = {
    # 3 = très agréable
    "forest",
    "grass",
    "meadow",
    "recreation_ground"
}

gdf.loc[gdf["landuse"].isin(landuse_agreable), "filtered"] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut conflits usage**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'conflit_md'

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1
gdf['severity_score'] = 0
gdf['severity_score'] = gdf['severity'].map({'high' : 3, 'medium':2, 'low':1})

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut service vélo**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'service_velo'

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")


# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut parking abris**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'parking_abris'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf["covered"] == "yes", "filtered"] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut accident**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----

attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gpd.read_file(
    f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
    encoding="latin1"   # or "ISO-8859-1"
)
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------

gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

**Attribut charge de trafic (TIM OFROU)**

Couche linéaire du modèle de trafic (plan de charges TI, jour moyen, état actuel), une ligne par lien et par sens. Valeur : `VOLVEHPRT(AP)` (tronquée en `VOLVEHPR~1` dans le .dbf), renommée `volvehprt_ap`.

Avec `TWO_WAY = True`, les deux sens d'un même lien (`NO`, géométries inversées) sont additionnés : on obtient la charge des deux sens sur une seule ligne. Sinon, la moyenne en 2_2 donnerait la charge d'un seul sens.

Agrégation en 2_2 : `how_GG = rectangle_weighted_mean` (`feature_rectangle.py`). Rectangle à extrémités plates autour du segment, demi-largeur `buffer_size_GG`, moyenne des lignes pondérée par leur longueur dans le rectangle ; les lignes obliques (> 45°) sont écartées.

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'charge_trafic'
TWO_WAY = True                     # additionner les deux sens d'un même lien (NO)
VALUE_COLUMN = "volvehprt_ap"      # nom de sortie -> value_column_GG de l'Excel
###--------------------

row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
if pd.isna(row[attribute_source_path]) or row[attribute_source_path] == 'unavailable':
    print(f"{attribute} non disponible pour le territoire {territory}")
else:
    path = f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}"
    gdf = gpd.read_file(path)   # CRS du .prj : Sphere_Mercator -> reprojeté plus bas
    print(f"Processing attribute: {attribute} ({len(gdf):,} liens orientés)")

    # Colonne de charge : VOLVEHPRT(AP), tronquée par le format .dbf (VOLVEHPR~1)
    vol_cols = [c for c in gdf.columns if c.upper().startswith("VOLVEHPR")]
    if len(vol_cols) != 1:
        raise ValueError(f"Colonne VOLVEHPRT(AP) introuvable ou ambiguë : {list(gdf.columns)}")
    gdf = gdf.rename(columns={vol_cols[0]: VALUE_COLUMN})
    gdf[VALUE_COLUMN] = pd.to_numeric(gdf[VALUE_COLUMN], errors="coerce")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].to_crs(operation_crs)

    if TWO_WAY:
        # Un lien = deux lignes (une par sens, géométrie inversée) : on somme les sens
        n_sens = gdf.groupby("NO").size()
        print(f"  Liens à deux sens : {int((n_sens == 2).sum()):,} | à un sens : {int((n_sens == 1).sum()):,}")
        gdf = gdf.groupby("NO", as_index=False).agg(
            **{VALUE_COLUMN: (VALUE_COLUMN, "sum"),
               "n_sens": (VALUE_COLUMN, "size"),
               "TYPENO": ("TYPENO", "first"),
               "geometry": ("geometry", "first")})
        gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=operation_crs)
    gdf = gdf.to_crs(target_crs)

    # Filter
    gdf['filtered'] = 0
    ###----change here----------------------
    gdf.loc[gdf[VALUE_COLUMN].notna(), 'filtered'] = 1
    ###--------------------------------------
    print("Filters applied")

    print(f"Proportion of features {attribute} kept after filtering:")
    print(gdf.filtered.value_counts(normalize=True))
    print(f"\n{VALUE_COLUMN} (véh./jour{', deux sens' if TWO_WAY else ', par sens'}) :")
    print(gdf[VALUE_COLUMN].describe().round(0))

    # Save
    save(save_filtered_attributes, row, gdf, attribute)

**Attribut Pente** !!! Depreciated 28.05.26 - cet attribut est maintenant calculé dans 1_1, directement sur le réseau, comme la cennectivité

In [ ]:
# # Initialize
# attribute = None
# row = None
# gdf = None
# print("Data initialized")

# # Load
# ###----change here-----
# attribute = 'pente'

# ###--------------------

    
# row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
# raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

# with rasterio.open(raster_path) as src:
#     # Read the raster data
#     raster_data = src.read(1)  # Read first band
    
#     # Define NoData
#     nodata_val = src.nodata if src.nodata is not None else -9999
    
#     # Valid mask: remove NoData
#     valid_mask = (raster_data != nodata_val)
    
#     # Get coordinates of valid points
#     rows, cols = np.where(valid_mask)
    
#     # Sample every 10th point
#     rows, cols = rows[::10], cols[::10]
    
#     # Get coordinates in map units
#     xs, ys = rasterio.transform.xy(src.transform, rows, cols)
#     temp_values = raster_data[rows, cols]
    
#     # Create GeoDataFrame directly with filtered points
#     gdf = gpd.GeoDataFrame({
#         'slope_degree': temp_values,
#         'filtered': 1,  # All points are filtered since we pre-filtered the data
#         'geometry': [Point(x, y) for x, y in zip(xs, ys)]
#     }, geometry='geometry')
    
#     # Set CRS and convert to target CRS
#     gdf = gdf.set_crs(src.crs)
#     gdf = gdf.to_crs(target_crs)

# print(f"Processing attribute: {attribute}")
# print("\nDescriptive statistics:")
# print(gdf['slope_degree'].describe())

# # Save
# print(f"Saving {attribute}: can take up to 3mn")
# save(save_filtered_attributes, row, gdf, attribute)

**Attribut tourner à droite**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tourner_droite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------

gdf.loc[gdf['TYPE_FEUX'] == 'plaq-comp-droite', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)